# Práctica 2. Herramientas de análisis de imagen de IA: YOLO

**PAI · Proyectos en Aplicaciones de Imagen · UPNA** · lunes 5 de octubre

## Qué vas a hacer hoy
Partiendo de un modelo ya entrenado, vas a detectar objetos en imágenes y en vídeo, y a seguir a cada objeto a lo largo del tiempo. El objetivo es entender **qué devuelve un detector, de qué parámetros depende y dónde falla**.

## Cómo se trabaja
- Ejecuta las celdas **en orden**, con `Shift + Enter`, y **lee siempre la salida**.
- Donde ponga `Escribe aquí el código…`, complétalo tú antes de ejecutar.
- Las celdas marcadas con ✍️ son **preguntas**: responde en la propia celda de texto, debajo del `>`.
  Doble clic para editar, `Shift + Enter` para volver a verla con formato. Forman parte de la entrega.


## Antes de empezar: el repositorio

Clona el repositorio de las prácticas y **crea tu propia rama** con "git switch -c", para que el trabajo de cada uno vaya por separado:

```powershell
git clone https://github.com/UPNa-Biomedical-Engineering-Group/PAI_practicas_2_3.git
cd PAI_practicas_2_3
git switch -c nombre-apellido 
```

Todo lo que hagas hoy se sube a esa rama. Al final de la práctica está el comando para entregarlo.

## 0 · Preparación del entorno

Las librerías se instalan en un **entorno virtual**: una carpeta con su propia copia de Python.

Para ello, desde el terminal de VS Code
(**Terminal → New Terminal**), dentro de la carpeta del repositorio:

```powershell
py -m venv .venv                  # crear (una sola vez)
.venv\Scripts\Activate.ps1        # activar (cada vez que abres un terminal)
pip install -r ..\requirements.txt   # instalar las librerías
```

El *prompt* pasa a empezar por `(.venv)`: esa es la señal de que estás dentro. Para salir, `deactivate`.

Después, arriba a la derecha de este cuaderno, **Select Kernel** → el que ponga `.venv`.

Si no aparece, Select Another Kernel > Python Environments > Create python environment > Enter interpreter path... > (Buscar la carpeta con .venv\Scripts\python.exe)



In [ ]:
import sys

import cv2
import numpy as np
from matplotlib import pyplot as plt

print("Python en uso:", sys.executable)
print("¿Es el del entorno virtual?", ".venv" in sys.executable or "envs" in sys.executable)
print("OpenCV:", cv2.__version__)


def mostrar(imagen_bgr, titulo="", ancho=10):
    """Muestra una imagen de OpenCV (BGR) dentro del cuaderno."""
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    alto = ancho * imagen_rgb.shape[0] / imagen_rgb.shape[1]

    plt.figure(figsize=(ancho, alto))
    plt.imshow(imagen_rgb)
    plt.title(titulo)
    plt.axis("off")
    plt.show()

## 1. YOLO en imagen

#### Generación del modelo YOLO


**YOLO** (*You Only Look Once*) es una familia de detectores de objetos. Su idea es mirar la imagen
**una sola vez**, con una única pasada de red, y predecir a la vez dónde está cada objeto y de qué
clase es. Por eso es rápido y se puede usar en vídeo.




### Cargar el modelo

El archivo `yolo26n.pt` contiene los **pesos** de un modelo ya entrenado sobre el conjunto COCO, que tiene 80 clases (persona, coche, autobús, perro…). La `n` es de *nano*: es la versión más pequeña y rápida de la familia, la que mejor va sin GPU.

La `n` del nombre es de *nano*: es la variante más pequeña y rápida de la familia.

`model_yolo` es un objeto al que se le pasan imágenes y devuelve detecciones.

In [ ]:
from ultralytics import YOLO

model_yolo = YOLO("yolo26n.pt")

#### Detección en una imagen

### Detección en una imagen

1. **`model_yolo(ruta)`** ejecuta la red sobre la imagen. Internamente la reescala, la normaliza, la pasa por la red y filtra las detecciones solapadas.
2. Devuelve una **lista** de resultados, una por imagen analizada. Como aquí solo pasamos una,
   nos interesa `results[0]`.
3. **`.plot()`** devuelve una copia de la imagen con las cajas y las etiquetas ya dibujadas.

Fíjate también en la línea de texto que imprime Ultralytics: resume lo que ha encontrado y lo que
ha tardado la inferencia.

In [ ]:
# Análisis en una imagen

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

results = model_yolo("images/bus.jpg")

mostrar(results[0].plot())

### ✍️ Pregunta 

Observa la imagen analizada. ¿Qué objetos ha detectado YOLO? ¿Están todos los que hay en la imagen?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### Qué hay dentro de un resultado

Todo lo detectado está en `result.boxes`, donde cada atributo es un **tensor con una fila por objeto**:

| Atributo | Contiene |
|---|---|
| `boxes.xyxy` | `[x1, y1, x2, y2]`: posición de la bounding box
| `boxes.cls` | El número de la clase.|
| `boxes.conf` | La confianza, entre 0 y 1 |
| `model_yolo.names` | El diccionario que traduce el número de clase a su nombre |

En la siguiente celda, el `zip` recorre los tres tensores a la vez, de modo que en cada vuelta tienes la caja, la clase y la confianza **del mismo objeto**. 


In [ ]:
# Análisis de la estructura de datos de los resultados

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

result = results[0]

boxes = result.boxes

for box, cls, conf in zip(
    boxes.xyxy,
    boxes.cls,
    boxes.conf
):
    class_id = int(cls)
    class_name = model_yolo.names[class_id]
    
    print(
        f"Clase: {class_id} | "
        f"Nombre: {class_name:15s} | "
        f"Confianza: {float(conf):.3f} | "
        f"Box: {box.tolist()}"
    )

### ✍️ Pregunta 

Observa las coordenadas `xyxy` de una detección. ¿Qué representan los cuatro valores?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

#### Lo que un detector preentrenado no puede ver

`yolo26n.pt` está entrenado sobre COCO, que tiene 80 clases de fotografía cotidiana. Prueba ahora con
dos imágenes distintas y observa el resultado antes de leer la pregunta del guion.

In [ ]:
for imagen in ["images/ciudad.jpg", "images/playa.jpg"]:
    resultado = model_yolo(imagen, conf=0.05,  verbose=False)[0]
    mostrar(resultado.plot(), f"{imagen} → {len(resultado.boxes)} objetos", ancho=6)

### ✍️ Pregunta 4

Con estas dos imágenes no detecta **nada**, ni bajando la confianza. Mira las imágenes y la lista de clases del principio y explica por qué. 

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

#### Efecto de los parámetros en el análisis

### El umbral de confianza

Cada detección lleva asociada una confianza. El parámetro **`conf`** descarta todo lo que quede por debajo del umbral; por defecto vale 0,25.

- Umbral **bajo**: aparecen más objetos, pero también más falsos positivos.
- Umbral **alto**: aparece solo lo seguro, a costa de perder objetos reales.

No hay un valor universalmente bueno: depende de la aplicación

In [ ]:
# Análisis del efecto de la confianza en los resultados

import time

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

confianzas = [0.25, 0.75]
start = []
elapsed = []
fps = []
n_objects = []

# Una vuelta por cada umbral; se cronometra cada llamada
for confianza in confianzas:
    start.append(time.perf_counter())


    results = model_yolo(
        "images/bus.jpg",
        conf=confianza
    )

    elapsed.append(time.perf_counter() - start[-1])
    fps.append(1 / elapsed[-1])
    mostrar(results[0].plot())

    n_objects.append(len(results[0].boxes))

# Imprimimos una tabla con los resultados de tiempo y FPS
print(
    f"{'Confianza':<15} {'Tiempo (ms)':<15} {'FPS':<10} {'Objetos':<10}\n"
    f"{'-'*50}\n"
)
for i, (confianza, t, f) in enumerate(zip(confianzas, elapsed, fps)):
    print(
        f"{confianza:<15} {t*1000:<15.6f} {f:<10.2f} {n_objects[i]:<10}"
    )

### ✍️ Pregunta 

Compara los resultados con los dos valores de confianza. ¿Qué ocurre al subir el umbral?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### ✍️ Pregunta 

¿Qué relación observas entre el número de detecciones y el tiempo de ejecución?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### El tamaño de entrada

Antes de analizar nada, YOLO **reescala la imagen a un cuadrado de lado `imgsz`**, 640 píxeles por
defecto. Es decir, el detector no ve tu imagen original, sino esa versión reducida.


Aquí está el compromiso central de la detección en vídeo: precisión contra velocidad.

In [ ]:
# Análisis del efecto del tamaño de la imagen en los resultados

import time

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

tamanos = [320, 640, 1280]


model_yolo("images/bus.jpg", verbose=False)
start = []
elapsed = []
fps = []
n_objects = []

for tamano in tamanos:
    start.append(time.perf_counter())

    '''
    -------------------------------------------------------------------------------------------
    Escribe aquí el código para realizar la detección de objetos en la imagen "images/bus.jpg"
    con el modelo YOLO y el tamaño de imagen especificado por la variable "tamano". La salida
    debe recogerse en la variable "results".
    -------------------------------------------------------------------------------------------
    '''
    results = model_yolo(
        "images/bus.jpg",
        imgsz=tamano
    )

    elapsed.append(time.perf_counter() - start[-1])
    fps.append(1 / elapsed[-1])
    mostrar(results[0].plot())

    n_objects.append(len(results[0].boxes))

# Imprimimos una tabla con los resultados de tiempo y FPS
print(
    f"{'Tamaño':<15} {'Tiempo (ms)':<15} {'FPS':<10} {'Objetos':<10}\n"
    f"{'-'*50}\n"
)
for i, (tamano, t, f) in enumerate(zip(tamanos, elapsed, fps)):
    print(
        f"{tamano:<15} {t*1000:<15.6f} {f:<10.2f} {n_objects[i]:<10}"
    )

### ✍️ Pregunta 

¿Qué efecto tiene aumentar el tamaño de la imagen sobre el número de objetos detectados, el tiempo de ejecución y los FPS (frames per second, número de imágenes que puedes procesar en un segundo)?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### ✍️ Pregunta 

¿Por qué un objeto pequeño es más fácil de detectar con una imagen de mayor resolución?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

## 2. YOLO en vídeo

#### Detección en un vídeo frame a frame

### Un vídeo es una secuencia de imágenes

A continuación se abre un archivo de vídeo y se leerán las imágenes una a una para pasarselas al modelo. Cada imagen se analiza por separado. 

> ⚠️ **Esta celda abre una ventana aparte**, fuera de VS Code: puede quedar detrás de otras ventanas (mira en la barra de tareas). 
> Para cerrarla, pulsa **Q** o **Esc** dentro de la ventana; no uses la X.

In [ ]:
# Detección en un vídeo frame a frame, con visualización de resultados

import cv2

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

# 1. Abrir el vídeo
cap = cv2.VideoCapture("images/calle.mp4")

# 2. Recorrer las imágenes una a una
while True:

    ret, frame = cap.read()

    if not ret:
        break

    '''
    --------------------------------------------------------------------------------------
    Escribe aquí el código para realizar la detección de objetos en el frame del vídeo
    con el modelo YOLO. La salida debe recogerse en la variable "results".
    --------------------------------------------------------------------------------------
    '''
    results = model_yolo(frame, verbose=FALSE)

    annotated = results[0].plot()

    cv2.imshow("YOLO", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# 4. Cerrar el vídeo y liberar la ventana
cap.release()
cv2.destroyAllWindows()

### ✍️ Pregunta 

¿Qué diferencia hay entre analizar una imagen y analizar un vídeo frame a frame?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### ✍️ Pregunta 

También se podría analizar una de cada N imágenes. ¿Qué ventajas y qué limitaciones tiene analizar todas frente a analizar solo algunas?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

#### Detección en un vídeo completo

Ultralytics sabe recorrer un vídeo por su cuenta pasandole la ruta. Con **`save=True`**
escribe el vídeo con las detecciones ya dibujadas en la carpeta `runs/detect/predict…`, en formato AVI.


In [ ]:
# Análisis de un vídeo

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

results = model_yolo("images/calle.mp4", save=True)

### ✍️ Pregunta 

¿Observas alguna diferencia entre el vídeo obtenido así y la colección de imágenes de la actividad anterior?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

## 3. Tracking YOLO

#### Tracking YOLO en un vídeo frame a frame

### De detectar a seguir

`model.track()` hace lo mismo que `model()` y además asocia las detecciones de imágenes
consecutivas y le pone a cada objeto un **identificador** que se mantiene mientras siga visible. El parámetro `persist=True` es el que le dice que recuerde los objetos entre llamadas.

Con esto ya se puede contar cuántos objetos distintos han aparecido, medir cuánto dura cada uno en escena o dibujar su trayectoria.

La celda siguiente tiene las siguientes funciones:

1. Preparación: abrir el vídeo, contar imágenes, crear la ventana.
2. Bucle principal por imagen: leerla y pasarla por el modelo `model.track()`.
3. Dibujo: `plot()` más el número de imagen sobreimpreso.
4. Teclado: **espacio** reproduce o pausa, **A** y **D** van atrás y adelante, **Q** o **Esc** salen.
5. Cierre ordenado en el `finally`, para que la ventana no quede colgada si algo falla.

El `try/finally` está para eso último: pase lo que pase dentro, el vídeo se cierra.

> ⚠️ **Esta celda abre una ventana aparte**, fuera de VS Code: puede quedar detrás de otras ventanas (barra de tareas). Para cerrarla, pulsa **Q** o **Esc** dentro de la ventana; no uses la X.

In [ ]:
# Tracking YOLO de un vídeo frame a frame con visualización de resultados y control de reproducción

import cv2

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

video_path = "images/calle.mp4"
window_name = "YOLO Tracking"

cv2.destroyAllWindows()

# 1. Preparación: vídeo y ventana
cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_id = 0
playing = False

cv2.namedWindow(window_name)

try:
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()

        if not ret:
            break

      
        results = model_yolo.track(
            frame,
            persist=True
        )
        annotated = results[0].plot()

        # 3. Dibujo: número de imagen sobre la imagen anotada
        cv2.putText(
            annotated,
            f"Frame: {frame_id + 1}/{total_frames}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.imshow(window_name, annotated)

        # 4. Teclado: espacio, A, D, Q/Esc
        key = cv2.waitKeyEx(30)
        key_low = key & 0xFF

        # Cerrar con la X
        try:
            if cv2.getWindowProperty(
                window_name,
                cv2.WND_PROP_VISIBLE
            ) < 1:
                break
        except cv2.error:
            break

        # ESC o Q
        if key == 27 or key_low == ord("q"):
            break

        # ESPACIO: reproducir/pausar
        elif key_low == ord(" "):
            playing = not playing

        # A: frame anterior
        elif key_low == ord("a"):
            playing = False
            frame_id = max(0, frame_id - 1)

        # D: frame siguiente
        elif key_low == ord("d"):
            playing = False
            frame_id = min(total_frames - 1, frame_id + 1)

        # Reproducción automática
        if playing:
            frame_id += 1

            if frame_id >= total_frames:
                frame_id = total_frames - 1
                playing = False

# 5. Cierre ordenado pase lo que pase
finally:
    cap.release()
    cv2.destroyAllWindows()

print("Reproductor de tracking cerrado.")

### ✍️ Pregunta 

Observa el vídeo con seguimiento. ¿Qué representa el ID que aparece junto a cada objeto?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### ✍️ Pregunta 

¿Por qué hace falta un sistema de seguimiento si ya detectamos personas en cada imagen?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### ✍️ Pregunta 

Busca un momento en el que dos objetos se crucen o se tapen. ¿Qué pasa con sus IDs?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### Lo que añade el seguimiento

Es la misma estructura de antes más un campo: **`boxes.id`**, el identificador de seguimiento.

- **`cls`** es la **clase**: qué tipo de objeto es. Todas las personas comparten `cls = 0`.
- **`id`** es la **instancia**: *qué* persona concreta es. Cada una tiene el suyo.

`boxes.id` puede valer `None`. Ocurre cuando el seguidor todavía no ha confirmado ningún objeto:
necesita ver **imágenes consecutivas** para asociarlas, así que en la primera imagen, o si saltas
de una imagen suelta a otra, no hay identificadores que dar.

In [ ]:
# Análisis de la estructura de datos de los resultados del tracking

result = results[0]

print("Estructura de boxes:")
print(result.boxes)
print("\n-----------------------------\n")
print("IDs de los objetos detectados:")
print(result.boxes.id)
print("\n-----------------------------\n")
print("Estadísticas de las detecciones:")
print("Coordenadas de los cuadros delimitadores (xyxy):")
print(result.boxes.xyxy)
print("Confianza de las detecciones:")
print(result.boxes.conf)
print("Clases de los objetos detectados:")
print(result.boxes.cls)

#### Demostración offline

### Guardar las trayectorias

Hasta ahora, al cerrar la ventana perdíamos la información del tracking. 
A continuación, vamos a recorrer el vídeo **sin mostrar nada**,
que es mucho más rápido, y guardar en disco lo que interesa:

- Un **JSON** con, por cada identificador, su clase, la imagen en la que apareció por primera vez y
  la lista de posiciones: imagen, centro y caja.
- Un **recorte** en `object_crops/` la primera vez que ve a cada objeto, para poder reconocerlo luego.


In [ ]:
# Detección YOLO en un vídeo frame a frame con tracking y guardado de trayectorias
# La visualización de los resultados se realiza en otra celda, con control de reproducción y selección de objetos

import cv2
import json
import os

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

video_path = "images/calle.mp4"
output_path = "calle_trajectories.json"
crops_dir = "object_crops"

os.makedirs(crops_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

# Diccionario: un identificador → su clase, su recorte y sus posiciones
trajectories = {}
frame_id = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    
    results = model_yolo.track(
        frame,
        persist=True,
        verbose=False,
        classes=[0, 2]  # 0: persona, 2: coche
    )
    
    result = results[0]
    boxes = result.boxes

    # Sin identificadores todavía no hay nada que guardar
    if boxes.id is not None:
        for box, track_id, cls in zip(
            boxes.xyxy,
            boxes.id,
            boxes.cls
        ):
            track_id = int(track_id)
            class_id = int(cls)

            x1, y1, x2, y2 = map(int, box.tolist())

            height, width = frame.shape[:2]
            x1 = max(0, min(x1, width - 1))
            y1 = max(0, min(y1, height - 1))
            x2 = max(0, min(x2, width))
            y2 = max(0, min(y2, height))

            if x2 <= x1 or y2 <= y1:
                continue

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            if track_id not in trajectories:
                crop_filename = f"object_{track_id}.jpg"
                crop_path = os.path.join(crops_dir, crop_filename)

                crop = frame[y1:y2, x1:x2]
                # Primera vez que vemos este objeto: guardamos su recorte
                cv2.imwrite(crop_path, crop)

                trajectories[track_id] = {
                    "class_id": class_id,
                    "class_name": model_yolo.names[class_id],
                    "first_frame": frame_id,
                    "crop_path": crop_path,
                    "positions": []
                }

            trajectories[track_id]["positions"].append({
                "frame": frame_id,
                "x": center_x,
                "y": center_y,
                "box": [x1, y1, x2, y2]
            })

    frame_id += 1

cap.release()

trajectories_json = {
    str(track_id): data
    for track_id, data in trajectories.items()
}

with open(output_path, "w", encoding="utf-8") as file:
    # Todo a disco, para no repetir el análisis
    json.dump(trajectories_json, file, indent=4)

print(f"Frames procesados: {frame_id}")
print(f"Objetos encontrados: {len(trajectories)}")
print(f"Recortes guardados en: {crops_dir}")
print(f"Trayectorias guardadas en: {output_path}")

### ✍️ Pregunta 

¿Por qué conviene guardar los resultados del seguimiento en un archivo, en lugar de volver a ejecutar YOLO cada vez que queremos ver una trayectoria?

*Escribe aquí tu respuesta (doble clic para editar, `Shift + Enter` para salir):*

> 

### El visualizador

Esta celda **no ejecuta ningún modelo**: solo carga el JSON y la lista de identificadores de la celda anterior y dibuja. 

Se dibuja la trayectoria completa de cada identificador, marcando la posición en la imagen actual y compone el panel de iconos en la parte superior. 
Con el teclado y ratón: **A** y **D** o clic para cambiar de objeto, **espacio** reproduce.


> ⚠️ **Esta celda abre una ventana aparte**, fuera de VS Code: puede quedar detrás de otras ventanas (barra de tareas).  Para cerrarla, pulsa **Q** o **Esc** dentro de la ventana; no uses la X.

In [ ]:
# Visualización del tracking YOLO con control de reproducción y selección de objetos
# Depende de la celda anterior, que genera el archivo "calle_trajectories.json" y los recortes de objetos en la carpeta "object_crops"

import cv2
import json
import os
import numpy as np

video_path = "images/calle.mp4"
trajectories_path = "calle_trajectories.json"
window_name = "Trayectoria del objeto"

with open(trajectories_path, "r", encoding="utf-8") as file:
    trajectories = json.load(file)

# 1. Qué objetos hay, leídos del JSON
object_ids = sorted(trajectories.keys(), key=int)

if not object_ids:
    raise ValueError("No se han encontrado objetos en trajectories.json")

selected_index = 0
frame_id = 0
playing = False
scroll_offset = 0

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ICON_WIDTH = 100
ICON_HEIGHT = 85
PANEL_HEIGHT = 110

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
visible_count = max(1, frame_width // ICON_WIDTH)
max_scroll = max(0, len(object_ids) - visible_count)


def slider_callback(value):
    global scroll_offset
    scroll_offset = min(value, max_scroll)


def change_object(index):
    global selected_index, frame_id

    selected_index = index % len(object_ids)
    frame_id = 0


def mouse_callback(event, x, y, flags, param):
    # Solo gestionar clics sobre los iconos.
    # No usamos la rueda del ratón.
    if event == cv2.EVENT_LBUTTONDOWN and y < PANEL_HEIGHT:
        clicked_index = (x // ICON_WIDTH) + scroll_offset

        if 0 <= clicked_index < len(object_ids):
            change_object(clicked_index)


# 2. Miniatura de un objeto, a partir de su recorte
def create_icon(object_id, selected):
    crop_path = trajectories[object_id]["crop_path"]
    icon = cv2.imread(crop_path)

    if icon is None:
        icon = np.zeros((70, 90, 3), dtype="uint8")
    else:
        scale = min(
            80 / icon.shape[1],
            65 / icon.shape[0]
        )

        icon = cv2.resize(
            icon,
            (
                max(1, int(icon.shape[1] * scale)),
                max(1, int(icon.shape[0] * scale))
            )
        )

    canvas = np.zeros((ICON_HEIGHT, ICON_WIDTH, 3), dtype="uint8")

    y_offset = (ICON_HEIGHT - icon.shape[0]) // 2
    x_offset = (ICON_WIDTH - icon.shape[1]) // 2

    canvas[
        y_offset:y_offset + icon.shape[0],
        x_offset:x_offset + icon.shape[1]
    ] = icon

    color = (0, 255, 0) if selected else (150, 150, 150)

    cv2.rectangle(
        canvas,
        (0, 0),
        (ICON_WIDTH - 1, ICON_HEIGHT - 1),
        color,
        3
    )

    cv2.putText(
        canvas,
        f"ID {object_id}",
        (8, ICON_HEIGHT - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1
    )

    return canvas


cv2.namedWindow(window_name)
cv2.setMouseCallback(window_name, mouse_callback)

# Slider para desplazar la lista de iconos
cv2.createTrackbar(
    "Objetos",
    window_name,
    0,
    max(1, max_scroll),
    slider_callback
)

try:
    while True:
        selected_id = object_ids[selected_index]
        selected_object = trajectories[selected_id]
        positions = selected_object["positions"]

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()

        if not ret:
            break

        # Dibujar la trayectoria completa
        points = [
            (int(position["x"]), int(position["y"]))
            for position in positions
        ]

        # 3. La trayectoria: una línea entre cada par de posiciones consecutivas
        for point_a, point_b in zip(points, points[1:]):
            cv2.line(
                frame,
                point_a,
                point_b,
                (0, 255, 255),
                3
            )

        # Buscar la posición del objeto en el frame actual
        current_position = next(
            (
                position
                for position in positions
                if position["frame"] == frame_id
            ),
            None
        )

        if current_position is not None:
            current_point = (
                int(current_position["x"]),
                int(current_position["y"])
            )

            # Punto verde del frame actual
            cv2.circle(
                frame,
                current_point,
                8,
                (0, 255, 0),
                -1
            )

            x1, y1, x2, y2 = current_position["box"]

            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (0, 255, 0),
                2
            )

        cv2.putText(
            frame,
            f"ID: {selected_id} | "
            f"Clase: {selected_object['class_name']}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Frame: {frame_id + 1}/{total_frames}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Crear panel superior de iconos
        # 4. Panel superior con las miniaturas
        panel = np.zeros(
            (PANEL_HEIGHT, frame.shape[1], 3),
            dtype="uint8"
        )

        visible_ids = object_ids[
            scroll_offset:scroll_offset + frame.shape[1] // ICON_WIDTH
        ]

        for position, object_id in enumerate(visible_ids):
            object_index = scroll_offset + position

            icon = create_icon(
                object_id,
                object_index == selected_index
            )

            x_offset = position * ICON_WIDTH

            panel[
                10:10 + ICON_HEIGHT,
                x_offset:x_offset + ICON_WIDTH
            ] = icon

        cv2.putText(
            panel,
            "Rueda: desplazar | A/D o clic: cambiar objeto",
            (10, PANEL_HEIGHT - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (200, 200, 200),
            1
        )

        display = cv2.vconcat((panel, frame))
        cv2.imshow(window_name, display)

        key = cv2.waitKeyEx(30) & 0xFF

        # ESC o Q
        if key in (27, ord("q")):
            break

        # ESPACIO: reproducir/pausar
        elif key == ord(" "):
            playing = not playing

        # A: objeto anterior
        elif key == ord("a"):
            change_object(selected_index - 1)

            if selected_index < scroll_offset:
                scroll_offset = selected_index
                cv2.setTrackbarPos(
                    "Objetos",
                    window_name,
                    scroll_offset
                )

        # D: objeto siguiente
        elif key == ord("d"):
            change_object(selected_index + 1)

            if selected_index >= scroll_offset + visible_count:
                scroll_offset = selected_index - visible_count + 1
                scroll_offset = min(scroll_offset, max_scroll)

                cv2.setTrackbarPos(
                    "Objetos",
                    window_name,
                    scroll_offset
                )

        if playing:
            frame_id += 1

            if frame_id >= total_frames:
                frame_id = total_frames - 1
                playing = False

finally:
    cap.release()
    cv2.destroyAllWindows()

print("Visualizador cerrado.")

#### La imagen puente para la práctica 3

Guardamos los resultados del frame 90 del video. En la práctica 3 continuaremos desde aquí para segmentar con SAM, sin ejecutar YOLO de nuevo.


In [ ]:
# Guardado de los resultados del frame 90 para trabajar  con SAM en la práctica 3

import cv2
import json
import os

video_path = "images/calle.mp4"
frame_number = 90
frame_path = "images/calle_frame_90.jpg"
annotated_frame_path = "images/calle_frame_90_annotated.jpg"
results_path = "calle_frame_90_results.json"

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    cap.release()
    raise RuntimeError(f"No se pudo abrir el vídeo: {video_path}")

cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
ret, frame_90 = cap.read()
cap.release()

if not ret or frame_90 is None:
    raise RuntimeError(f"No se pudo leer el frame {frame_number}")

# Detectar personas en un frame aislado. Para una imagen no usamos tracking:
# el tracker puede descartar detecciones al no tener contexto temporal.
results_90 = model_yolo.track(
    frame_90,
    classes=[0],
    imgsz=1920,
    conf=0.01,
    verbose=False,
    persist=True
)[0]

boxes_90 = results_90.boxes
frame_90_boxes = []

for index, box in enumerate(boxes_90.xyxy.cpu().tolist()):
    class_id = int(boxes_90.cls[index].item())
    confidence = float(boxes_90.conf[index].item())
    track_id = None

    x1, y1, x2, y2 = map(int, box)
    frame_90_boxes.append({
        "index": index,
        "track_id": track_id,
        "class_id": class_id,
        "class_name": model_yolo.names[class_id],
        "confidence": confidence,
        "xyxy": [x1, y1, x2, y2],
        "center": [(x1 + x2) // 2, (y1 + y2) // 2]
    })

frame_90_metadata = {
    "video_path": video_path,
    "frame_number": frame_number,
    "image_path": frame_path,
    "annotated_image_path": annotated_frame_path,
    "image_width": int(frame_90.shape[1]),
    "image_height": int(frame_90.shape[0]),
    "boxes": frame_90_boxes
}

os.makedirs(os.path.dirname(frame_path), exist_ok=True)
cv2.imwrite(frame_path, frame_90)
cv2.imwrite(annotated_frame_path, results_90.plot())

with open(results_path, "w", encoding="utf-8") as file:
    json.dump(frame_90_metadata, file, indent=4)

# Variables reutilizables en las siguientes celdas, por ejemplo para SAM.
frame_90_rgb = cv2.cvtColor(frame_90, cv2.COLOR_BGR2RGB)
frame_90_boxes_xyxy = [item["xyxy"] for item in frame_90_boxes]

print(f"Frame guardado en: {frame_path}")
print(f"Resultados guardados en: {results_path}")
print(f"Personas detectadas: {len(frame_90_boxes)}")

---
# Entrega

Guarda el cuaderno (**Ctrl+S**) con todas las celdas ejecutadas y las preguntas respondidas, y súbelo
a tu rama:

```powershell
git add .
git commit -m "Entrega práctica YOLO"
git push origin nombre-apellido
```

La primera vez que subas la rama, Git te pedirá `git push --set-upstream origin nombre-apellido`:
es lo mismo, cópialo tal cual.

Entra después en GitHub y comprueba que tu rama aparece y que el commit está ahí.

